# BotHP (KDD 2025) trên MGTAB - runtime Colab RIÊNG
Repo: Peien429/BotHP. MGTAB chỉ có **một** lệnh: `python main_transductive.py --dataset MGTAB-FF` (followers + friends; `--use_cfg` mặc định True nên `configs.yml` luôn được nạp; 5 seed mặc định 3289, 3389, 3889, 7089, 7189). Repo **không có cờ chọn biến thể RGCN/RGT**: hỏi Sơn ý của TASKS.md.

Repo pin `torch==1.13.0`, `torch-geometric==2.1.0`, `scipy 1.10.1`, `pandas 1.5.3` -> cần **Python 3.10** (Colab hiện là 3.13), dùng venv `uv`. `requirements.txt` **thiếu `faiss`** (models/pretrain_model.py import faiss) -> cài thêm `faiss-cpu`.

Dữ liệu: `process.py` đọc `features.pt`, `edge_index.pt`, `edge_type.pt`, `label.pt`; bộ của nhóm có `labels_bot.pt` nên chỉ cần copy thành `label.pt`. `process.py` tự chia **70/20/10 ngẫu nhiên (random_state=0)** và đổi chỗ `test_mask`/`val_mask` -> số BotHP ở đây chỉ để kiểm chứng số đã công bố, không so thẳng với `split()` của nhóm.

Ghi lại mọi lỗi cài đặt làm bằng chứng "đã thử nghiêm túc". Nếu không cài được: báo Sơn và dùng `SHGN.py`.

**Lưu ý (code BotHP cố ý hoán đổi val/test):** process.py chia 70/20/10 (seed 0), rồi gán
test_mask = khối 20% và val_mask = khối 10%. final_* và early-stopping_* của BotHP
được đo trên tập 20% (~2040 tài khoản), không phải 10% như split thông thường.
F1 trong bảng là binary F1 của lớp bot (node_classification_evaluation.py dòng 31).

In [ ]:
%cd /content
!pip install -q uv gdown
!test -d BotHP || git clone https://github.com/Peien429/BotHP.git
%cd /content/BotHP
!ls
!ls models
!grep -rn "RGCN\|RGT" models | head -20      # có biến thể backbone không?

/content
Cloning into 'BotHP'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 26 (delta 0), reused 26 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 13.10 KiB | 4.37 MiB/s, done.
/content/BotHP
configs.yml	      models				 requirements.txt
dataset		      node_classification_evaluation.py  utils.py
main_transductive.py  README.md
decoder.py  fintune_model.py  pretrain_model.py  utils.py
encoder.py  __init__.py       RGT.py
models/encoder.py:5:from models.RGT import RGTLayer
models/encoder.py:105:        self.layer1 = RGTLayer(num_edge_type=2, in_channel=num_hidden, out_channel=num_hidden, trans_heads=nhead,
models/encoder.py:107:        self.layer2 = RGTLayer(num_edge_type=2, in_channel=num_hidden, out_channel=num_hidden, trans_heads=nhead,
models/encoder.py:168:        self.layer1 = RGTLayer(num_edge_type=num_edge_type, in_channel=num_hidden, out_channel=num_hidden

## 1. Môi trường Python 3.10 (cài từng bước, chỉ dùng wheel dựng sẵn cho gói PyG)

In [ ]:
# 1a. Venv sạch + torch 1.13.0 (cu117) trước
!rm -rf /content/venv310
!uv venv --python 3.10 /content/venv310
!uv pip install --python /content/venv310/bin/python torch==1.13.0 torchvision==0.14.0 torchaudio==0.13.0 --extra-index-url https://download.pytorch.org/whl/cu117 --index-strategy unsafe-best-match
!/content/venv310/bin/python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

Using CPython 3.10.22
Creating virtual environment at: /content/venv310
Activate with: source /content/venv310/bin/activate
Using Python 3.10.22 environment at: /content/venv310
Resolved 11 packages in 678ms
Prepared 11 packages in 50.79s
Installed 11 packages in 310ms
 + certifi==2026.10.23
 + charset-normalizer==3.5.2
 + idna==3.20
 + numpy==2.2.6
 + pillow==12.3.0
 + requests==2.34.2
 + torch==1.13.0+cu117
 + torchaudio==0.13.0+cu117
 + torchvision==0.14.0+cu117
 + typing-extensions==4.16.0
 + urllib3==2.8.0
1.13.0+cu117 True


In [ ]:
# 1b. Xem chỉ mục PyG thực sự có wheel nào cho torch 1.13 + cu117 + Python 3.10
!curl -s https://data.pyg.org/whl/torch-1.13.0+cu117.html | grep -o 'torch_[a-z_]*-[0-9.]*+pt113cu117-cp310-cp310-linux_x86_64.whl' | sort -u

torch_cluster-1.6.0+pt113cu117-cp310-cp310-linux_x86_64.whl
torch_cluster-1.6.1+pt113cu117-cp310-cp310-linux_x86_64.whl
torch_scatter-2.1.0+pt113cu117-cp310-cp310-linux_x86_64.whl
torch_scatter-2.1.1+pt113cu117-cp310-cp310-linux_x86_64.whl
torch_sparse-0.6.15+pt113cu117-cp310-cp310-linux_x86_64.whl
torch_sparse-0.6.16+pt113cu117-cp310-cp310-linux_x86_64.whl
torch_sparse-0.6.17+pt113cu117-cp310-cp310-linux_x86_64.whl
torch_spline_conv-1.2.1+pt113cu117-cp310-cp310-linux_x86_64.whl
torch_spline_conv-1.2.2+pt113cu117-cp310-cp310-linux_x86_64.whl


In [ ]:
# 1c. Cài 4 gói mở rộng PyG CHỈ từ wheel (không build từ nguồn). Nếu index chỉ có bản thấp hơn requirements pin
# thì dùng bản đó và ghi chú khác biệt. Nếu thiếu torch-cluster, thử bỏ nó khỏi lệnh.
!uv pip install --python /content/venv310/bin/python --only-binary :all: --no-deps --find-links https://data.pyg.org/whl/torch-1.13.0+cu117.html torch-scatter torch-sparse torch-cluster torch-spline-conv

Using Python 3.10.22 environment at: /content/venv310
Resolved 4 packages in 95ms
Prepared 4 packages in 537ms
Installed 4 packages in 6ms
 + torch-cluster==1.6.1+pt113cu117
 + torch-scatter==2.1.1+pt113cu117
 + torch-sparse==0.6.17+pt113cu117
 + torch-spline-conv==1.2.2+pt113cu117


In [ ]:
# 1d. Phần còn lại của requirements.txt (bỏ mkl-service và các gói torch đã cài) + faiss-cpu (requirements thiếu)
!grep -v -E "^(mkl-service|torch==|torch-cluster|torch-scatter|torch-sparse|torch-spline-conv|torchaudio|torchvision)" requirements.txt > req_rest.txt
!uv pip install --python /content/venv310/bin/python -r req_rest.txt
!uv pip install --python /content/venv310/bin/python faiss-cpu==1.7.4
!/content/venv310/bin/python -c "import torch, torch_geometric, torch_scatter, torch_sparse, faiss; print(torch.__version__, torch_geometric.__version__, faiss.__version__, torch.cuda.is_available())"

Using Python 3.10.22 environment at: /content/venv310
Resolved 48 packages in 2.04s
Prepared 40 packages in 6.27s
Uninstalled 1 package in 43ms
Installed 41 packages in 216ms
 + aiohttp==3.9.5
 + aiosignal==1.3.1
 + async-timeout==4.0.3
 + attrs==23.2.0
 + click==8.1.7
 + docker-pycreds==0.4.0
 + frozenlist==1.4.1
 + fsspec==2024.5.0
 + gitdb==4.0.11
 + gitpython==3.1.43
 + jinja2==3.1.2
 + joblib==1.3.2
 + lightning-utilities==0.11.2
 + markupsafe==2.1.3
 + multidict==6.0.5
 + networkx==3.1
 - numpy==2.2.6
 + numpy==1.26.4
 + packaging==24.0
 + pandas==1.5.3
 + platformdirs==4.3.6
 + protobuf==5.28.2
 + psutil==6.0.0
 + pygcl==0.1.2
 + pyparsing==3.1.1
 + python-dateutil==2.9.0.post0
 + pytorch-lightning==2.2.5
 + pytz==2024.2
 + pyyaml==6.0.1
 + scikit-learn==1.3.2
 + scipy==1.10.1
 + sentry-sdk==2.14.0
 + setproctitle==1.3.3
 + setuptools==84.2.1
 + six==1.16.0
 + smmap==5.0.1
 + threadpoolctl==3.2.0
 + torch-geometric==2.1.0
 + torchmetrics==1.4.0.post0
 + tqdm==4.66.1
 + tzdata==2

## 2. Dữ liệu (bản MGTAB.zip của nhóm) -> `dataset/raw/MGTAB` -> `process.py`

In [ ]:
import os, glob, shutil, subprocess, zipfile
os.chdir("/content/BotHP")
os.makedirs("dataset/raw/MGTAB", exist_ok=True)
subprocess.run(["gdown", "1gbWNOoU1JB8RrTu2a5j9KMNVa9wX72Fe", "-O", "MGTAB.zip"], check=True)
zipfile.ZipFile("MGTAB.zip").extractall("_tmp")
hit = glob.glob("_tmp/**/features.pt", recursive=True)
assert hit, "Không thấy features.pt trong zip"
src = os.path.dirname(hit[0])
for f in os.listdir(src):
    shutil.copy(os.path.join(src, f), os.path.join("dataset/raw/MGTAB", f))
shutil.rmtree("_tmp", ignore_errors=True)
print(sorted(os.listdir("dataset/raw/MGTAB")))

['edge_index.pt', 'edge_type.pt', 'edge_weight.pt', 'features.pt', 'labels_bot.pt', 'labels_stance.pt', 'process.py']


In [ ]:
# process.py đọc 'label.pt'; bộ của nhóm có labels_bot.pt (nhãn bot) -> copy sang đúng tên rồi chạy.
%cd /content/BotHP/dataset/raw/MGTAB
!cp labels_bot.pt label.pt
!/content/venv310/bin/python process.py
%cd /content/BotHP
!ls -la dataset/processed       # phải có MGTAB-followers-friends.pt

/content/BotHP/dataset/raw/MGTAB
/content/BotHP
total 48412
drwxr-xr-x 2 root root     4096 Nov  6 12:19 .
drwxr-xr-x 4 root root     4096 Nov  6 12:17 ..
-rw-r--r-- 1 root root      746 Nov  6 12:17 community.py
-rw-r--r-- 1 root root 49558819 Nov  6 12:19 MGTAB-followers-friends.pt


## 3. Kiểm tra trước khi tin số

In [ ]:
%cd /content/BotHP
!grep -n "mask" node_classification_evaluation.py     # mask nào dùng để chọn model, mask nào để báo final_*?
!grep -n "f1\|average" node_classification_evaluation.py   # loại F1 (binary hay macro)?

/content/BotHP
54:    train_mask = graph.train_mask
55:    val_mask = graph.val_mask
56:    test_mask = graph.test_mask
65:        train_acc = compute_metrics(labels[train_mask], out[train_mask])['accuracy']
66:        loss = criterion(out[train_mask], labels[train_mask])
74:            val_metrics = compute_metrics(labels[val_mask], pred[val_mask])
77:            val_loss = criterion(pred[val_mask], labels[val_mask])
88:        final_metrics = compute_metrics(labels[test_mask], pred[test_mask])
92:        estp_metrics = compute_metrics(labels[test_mask], pred[test_mask])
6:from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, precision_recall_curve
12:        'f1': 0.0,
31:        metrics['f1'] = round(f1_score(y_true, y_pred_label), 5)
76:            val_f1 = val_metrics['f1']
84:            f' val_loss:{val_loss.item(): .4f}, val_acc:{val_acc}, val_f1:{val_f1}')


## 4. Chạy BotHP (5 seed mặc định)

In [ ]:
# Chỉ in kết quả tổng hợp ở cuối: final_* và early-stopping_*. Log đầy đủ ở bothp_run.log
%cd /content/BotHP
!/content/venv310/bin/python main_transductive.py --dataset MGTAB-FF 2>&1 | tee bothp_run.log | tail -25

/content/BotHP
# Epoch: 299, train_loss: 0.1408, train_acc:0.94229, val_loss: 0.2857, val_acc:0.89118, val_f1:0.80143: 100%|██████████| 300/300 [08:14<00:00,  1.65s/it]
--- Final-epoch-TestAcc: 0.9108, best-val-epoch-TestAcc: 0.9088, best-val-epoch: 203.
####### Run 2 for seed 3889
Namespace(seeds=[3289, 3389, 3889, 7089, 7189], dataset='MGTAB-FF', device=0, max_epoch=100, num_heads_encoder=4, num_hidden=128, feat_drop=0.4, attn_drop=0.3, finetune_drop=0.4, lr=0.0005, weight_decay=0.002, feat_mask_rate='0.4,+0.01,0.6', replace_rate=0.25, leave_unchanged_rate=0.15, max_epoch_f=300, lr_f=0.00025, weight_decay_f=0.0001, use_cfg=True, scheduler=True, cluster_num=20, interval=10, advanced_loss_rate=0.005, instance_prototype_loss_rate=1, optimizer='adam', num_features=788)
# Epoch 99: train_loss: 1.9893163443: 100%|██████████| 100/100 [03:39<00:00,  2.20s/it]
# Epoch: 299, train_loss: 0.1320, train_acc:0.94733, val_loss: 0.3019, val_acc:0.88824, val_f1:0.80139: 100%|██████████| 300/300 [08:1

Nếu lỗi `does not have a deterministic implementation`: `utils.py` bật `torch.use_deterministic_algorithms(True)`; dán lỗi lên, đừng tự sửa.

## Ghi kết quả (copy sang `reproduce.ipynb`, mục 5)
Điền loại F1 từ cell 3. Nếu là binary F1 thì tính lại macro-F1 từ dự đoán trước khi so với RGT.

| | accuracy | precision | recall | F1 (loại?) | ghi chú |
|---|---|---|---|---|---|
| final | | | | | |
| early-stopping | | | | | |

Khác biệt so với repo gốc: Python 3.10 venv + uv, bỏ mkl-service, cài thêm faiss-cpu, wheel PyG phiên bản: ..., dữ liệu của nhóm (labels_bot.pt copy thành label.pt).
Lỗi cài đặt đã gặp (nếu có): ...

In [ ]:
!grep -E "final_|early-stopping_" /content/BotHP/bothp_run.log

final_accuracy: 0.9101±0.0014
early-stopping_accuracy: 0.9102±0.0012
final_precision: 0.8514±0.0063
early-stopping_precision: 0.8457±0.0042
final_recall: 0.8048±0.0135
early-stopping_recall: 0.8128±0.0100
final_f1: 0.8273±0.0046
early-stopping_f1: 0.8289±0.0035


In [ ]:
!grep -n "f1\|average" /content/BotHP/node_classification_evaluation.py